# Investigación: Tensión R∞ vs GUE

**Pregunta:** ¿Por qué el dataset v6 (CLAUDE.md) da R∞ = 0.59884 ± 0.00012 (−7.4σ de GUE),
mientras que el análisis anterior (viaje_completo.md) daba R∞ = 0.59937 ± 0.00021 (−0.2σ)?

**GUE_REF = 0.59971** (Atas et al. 2013)

## Hipótesis a testear
1. ¿Es el `sigma_floor = 0.00020` artificial el culpable? (da peso extra a puntos bajos)
2. ¿Es el punto logT=20.2003 (r=0.60145, −2.3σ) un outlier estadístico que arrastra R∞?
3. ¿Son los puntos nuevos en logT 19–24 genuinamente más bajos que el modelo anterior?
4. ¿Es R∞ < GUE un resultado físico real, o un artefacto del sigma_floor?


In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

GUE_REF     = 0.59971   # Atas et al. 2013
SIGMA_FLOOR = 0.00020   # sigma_naive(N=500k) ≈ 0.00019

print(f'GUE_REF = {GUE_REF}')
print(f'SIGMA_FLOOR = {SIGMA_FLOOR}')

## 1. Dataset v6 completo (estado actual CLAUDE.md)

In [ ]:
# ── Puntos originales (logT < 19, N=50k) ─────────────────────────
logTs_orig = np.array([9.736, 10.003, 10.665, 12.432, 14.755, 15.997, 17.212, 18.412])
rs_orig    = np.array([0.61188, 0.61132, 0.61012, 0.60683, 0.60472, 0.60488, 0.60344, 0.60347])
sigma_orig = np.array([0.00060, 0.00060, 0.00060, 0.00029, 0.00060, 0.00094, 0.00076, 0.00048])
# Nota: logT 9.7/10.0/10.7 usan sigma_naive (fichero compartido zeros_26000.dat)

# ── Cuadrícula fina (logT 19–24, N=500k, 13 ficheros únicos) ────
logTs_grid     = np.array([19.0031, 19.2043, 19.4036, 19.6025, 19.8005,
                            20.0010, 20.2003, 20.3988, 20.4104,
                            21.0036, 22.0614, 23.1151, 24.1445])
rs_grid        = np.array([0.60265, 0.60215, 0.60208, 0.60203, 0.60196,
                            0.60221, 0.60145, 0.60187, 0.60180,
                            0.60169, 0.60154, 0.60126, 0.60101])
sigma_grid_emp = np.array([0.00014, 0.00022, 0.00018, 0.00024, 0.00028,
                            0.00044, 0.00013, 0.00016, 0.00030,
                            0.00029, 0.00031, 0.00030, 0.00023])

# sigma_floor: aplica cuando sigma_emp < 0.00020
sigma_grid_fl  = np.maximum(sigma_grid_emp, SIGMA_FLOOR)
floor_applied  = sigma_grid_emp < SIGMA_FLOOR  # 4 puntos

# ── Dataset combinado y ordenado ─────────────────────────────────
lt_all  = np.concatenate([logTs_orig, logTs_grid])
r_all   = np.concatenate([rs_orig,    rs_grid])
sig_fl  = np.concatenate([sigma_orig, sigma_grid_fl])   # con floor
sig_emp = np.concatenate([sigma_orig, sigma_grid_emp])  # sin floor
fl_all  = np.concatenate([np.zeros(8, bool), floor_applied])

idx = np.argsort(lt_all)
lt_all, r_all, sig_fl, sig_emp, fl_all = [x[idx] for x in [lt_all, r_all, sig_fl, sig_emp, fl_all]]

print(f'Total puntos: {len(lt_all)}')
print(f'Puntos con sigma_floor aplicado: {fl_all.sum()}')
print()
print(f'  {"logT":>8}  {"r":>9}  {"sig_emp":>8}  {"sig_fl":>8}  {"floor?":>6}')
print('  ' + '─'*50)
for i in range(len(lt_all)):
    flag = ' ← FLOOR' if fl_all[i] else ''
    print(f'  {lt_all[i]:>8.4f}  {r_all[i]:>9.5f}  {sig_emp[i]:>8.5f}  {sig_fl[i]:>8.5f}  {str(fl_all[i]):>6}{flag}')

## 2. Ajuste del modelo A en 6 escenarios distintos

Modelo A: `⟨r⟩(T) = R∞ + c / log²(T)`

In [ ]:
def model_A(x, R, c):
    return R + c / x**2

def fit_A(lt, r, sig, label=''):
    """Ajusta modelo A, devuelve dict con resultados."""
    p, cov = curve_fit(model_A, lt, r, sigma=sig, p0=[0.5994, 1.25], maxfev=20000)
    e      = np.sqrt(np.diag(cov))
    res    = r - model_A(lt, *p)
    chi2   = np.sum((res / sig)**2)
    dof    = len(lt) - 2
    nsig   = (p[0] - GUE_REF) / e[0]
    return dict(R=p[0], eR=e[0], c=p[1], ec=e[1],
                chi2=chi2, dof=dof, chi2_dof=chi2/dof,
                nsig_gue=nsig, res=res, n=len(lt), label=label)

# Dataset v1 (viaje_completo.md, 12 puntos)
lt_v1 = np.array([9.736, 10.003, 10.665, 12.432, 14.755, 15.997,
                   17.212, 18.412, 19.11, 19.81, 20.10, 24.15])
r_v1  = np.array([0.61188, 0.61132, 0.61012, 0.60683, 0.60472, 0.60488,
                   0.60344, 0.60347, 0.60202, 0.60178, 0.60299, 0.60140])
s_v1  = np.full(12, 0.00060)
s_v1[-1] = 0.00030

# 6 escenarios
escenarios = [
    fit_A(lt_all, r_all, sig_fl,
          'v6 sigma_floor (ACTUAL — CLAUDE.md)'),
    fit_A(lt_all, r_all, sig_emp,
          'v6 sigma_emp   (sin floor, 21 pts)'),
    fit_A(lt_all[~fl_all], r_all[~fl_all], sig_fl[~fl_all],
          'v6 excl. 4 pts floor (17 pts)'),
    fit_A(lt_all[lt_all < 19], r_all[lt_all < 19], sig_fl[lt_all < 19],
          'Solo logT<19 (8 pts originales)'),
    fit_A(lt_all[r_all != 0.60145], r_all[r_all != 0.60145], sig_fl[r_all != 0.60145],
          'v6 sin logT=20.2003 (20 pts)'),
    fit_A(lt_v1, r_v1, s_v1,
          'v1 viaje_completo (12 pts, sigma uniforme)'),
]

print('═'*80)
print('COMPARACIÓN DE ESCENARIOS — Modelo A: ⟨r⟩ = R∞ + c/log²T')
print('═'*80)
print(f'  {"Escenario":42s}  N   R∞          ±       nσ(GUE)     c       χ²/dof')
print('  ' + '─'*78)
for f in escenarios:
    print(f'  {f["label"]:42s}  {f["n"]:2d}  '
          f'{f["R"]:.5f} ± {f["eR"]:.5f}  {f["nsig_gue"]:>+7.1f}σ  '
          f'{f["c"]:.4f}  {f["chi2_dof"]:>7.3f}')

print()
print(f'  GUE_REF = {GUE_REF}')

## 3. Diagnóstico: efecto de cada punto con sigma_floor

In [ ]:
# Predicción del modelo v6 actual
f_base = escenarios[0]
pred   = model_A(lt_all, f_base['R'], f_base['c'])

print('PUNTOS CON SIGMA_FLOOR APLICADO — residuos con ambos errores')
print('─'*72)
print(f'  {"logT":>8}  {"r_obs":>9}  {"r_pred":>9}  '
      f'{"sig_emp":>8}  {"sig_fl":>8}  {"inflac":>7}  '
      f'{"res/emp":>9}  {"res/fl":>8}')
for i in range(len(lt_all)):
    if fl_all[i]:
        res = r_all[i] - pred[i]
        infl = sig_fl[i] / sig_emp[i]
        print(f'  {lt_all[i]:>8.4f}  {r_all[i]:>9.5f}  {pred[i]:>9.5f}  '
              f'{sig_emp[i]:>8.5f}  {sig_fl[i]:>8.5f}  {infl:>7.2f}x  '
              f'{res/sig_emp[i]:>+9.2f}σ  {res/sig_fl[i]:>+8.2f}σ')

print()
print('TODOS LOS RESIDUOS (modelo v6 sigma_floor):')
print('─'*56)
for i in range(len(lt_all)):
    res_norm = f_base['res'][i] / sig_fl[i]
    bar = '█' * int(abs(res_norm) * 5)
    sign = '+' if res_norm > 0 else '-'
    fl_mark = ' ← FLOOR' if fl_all[i] else ''
    print(f'  logT={lt_all[i]:7.4f}  {res_norm:>+6.2f}σ  {sign}{bar}{fl_mark}')

## 4. Test estadístico: ¿es logT=20.2003 un outlier genuino?

In [ ]:
# Test de Grubbs para outliers
# El punto logT=20.2003 tiene r=0.60145 con sigma_emp=0.00013

# Con el modelo v6 ajustado, el residuo en sigma_emp es:
i_20 = np.argmin(np.abs(lt_all - 20.2003))
res_emp  = f_base['res'][i_20] / sig_emp[i_20]
res_fl   = f_base['res'][i_20] / sig_fl[i_20]

# ¿Qué tan probable es ver un residuo de este tamaño?
from scipy.stats import norm
p_emp = 2 * norm.sf(abs(res_emp))   # two-sided p-value
p_fl  = 2 * norm.sf(abs(res_fl))

print('TEST: ¿Es logT=20.2003 un outlier estadístico?')
print('─'*55)
print(f'  r_obs    = {r_all[i_20]:.5f}')
print(f'  r_pred   = {pred[i_20]:.5f}  (modelo A v6)')
print(f'  residuo  = {f_base["res"][i_20]:+.5f}')
print()
print(f'  Con sigma_emp = {sig_emp[i_20]:.5f}:')
print(f'    residuo / sigma_emp = {res_emp:+.2f}σ')
print(f'    p-valor (dos colas) = {p_emp:.4f}  → probabilidad de ver esto por azar')
print()
print(f'  Con sigma_fl  = {sig_fl[i_20]:.5f}  (sigma_floor):')
print(f'    residuo / sigma_fl  = {res_fl:+.2f}σ')
print(f'    p-valor (dos colas) = {p_fl:.4f}')
print()

# Corrección de Bonferroni: con 21 puntos, el umbral de significancia es 0.05/21
bonf = 0.05 / len(lt_all)
print(f'  Corrección Bonferroni (21 tests): umbral = {bonf:.4f}')
print(f'  Con sigma_emp: p={p_emp:.4f} {'< umbral → OUTLIER SIGNIFICATIVO' if p_emp < bonf else '> umbral → no outlier significativo'}')
print(f'  Con sigma_fl:  p={p_fl:.4f} {'< umbral → OUTLIER SIGNIFICATIVO' if p_fl < bonf else '> umbral → no outlier significativo'}')

## 5. Análisis de influencia: ¿cuánto mueve R∞ cada punto individual?

In [ ]:
# Leave-one-out: ajustar modelo A quitando un punto a la vez
# Ver cuánto cambia R∞

R_base = f_base['R']

print('ANÁLISIS LEAVE-ONE-OUT: influencia de cada punto sobre R∞')
print('─'*65)
print(f'  R∞ base (todos 21 pts) = {R_base:.5f}')
print()
print(f'  {"logT":>8}  {"r":>9}  {"R∞ s/pt":>10}  {"ΔR∞":>9}  {"influencia":>10}  floor?')
print('  ' + '─'*63)

influences = []
for j in range(len(lt_all)):
    mask = np.ones(len(lt_all), bool)
    mask[j] = False
    try:
        fj = fit_A(lt_all[mask], r_all[mask], sig_fl[mask])
        delta = fj['R'] - R_base
        influences.append((j, lt_all[j], r_all[j], fj['R'], delta, fl_all[j]))
    except:
        influences.append((j, lt_all[j], r_all[j], np.nan, np.nan, fl_all[j]))

# Ordenar por |influencia|
influences.sort(key=lambda x: abs(x[4]) if not np.isnan(x[4]) else 0, reverse=True)
for j, lT, r, R_loo, delta, fl in influences:
    fl_str = 'FLOOR' if fl else ''
    bar = '▶' * min(int(abs(delta) / 0.00002), 20)
    sign = '+' if delta > 0 else '-'
    print(f'  {lT:>8.4f}  {r:>9.5f}  {R_loo:>10.5f}  {delta:>+9.5f}  {sign}{bar:20s}  {fl_str}')

## 6. Visualización: los 6 escenarios sobre los datos

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9),
                                 gridspec_kw={'height_ratios': [3, 1]},
                                 sharex=True)

lt_plot = np.linspace(9, 25, 500)

# Colores para escenarios
colors   = ['navy', 'royalblue', 'steelblue', 'gray', 'darkorange', 'green']
styles   = ['-', '--', '-.', ':', '-', '--']
alphas   = [1.0, 0.9, 0.8, 0.7, 0.9, 0.9]
lws      = [2.5, 2, 1.8, 1.5, 2, 2]

for esc, col, sty, al, lw in zip(escenarios, colors, styles, alphas, lws):
    yfit = model_A(lt_plot, esc['R'], esc['c'])
    ax1.plot(lt_plot, yfit, color=col, ls=sty, alpha=al, lw=lw,
             label=f"{esc['label']}\n  R∞={esc['R']:.5f} ({esc['nsig_gue']:+.1f}σ GUE)")

# Datos
mask_fl = fl_all
ax1.errorbar(lt_all[lt_all < 19], r_all[lt_all < 19],
             yerr=sig_fl[lt_all < 19], fmt='o', color='black',
             ms=6, capsize=4, zorder=5, label='datos orig (logT<19)')
ax1.errorbar(lt_all[(lt_all >= 19) & ~mask_fl], r_all[(lt_all >= 19) & ~mask_fl],
             yerr=sig_fl[(lt_all >= 19) & ~mask_fl], fmt='s', color='darkblue',
             ms=6, capsize=4, zorder=5, label='datos grid (logT≥19, sigma OK)')
ax1.errorbar(lt_all[mask_fl], r_all[mask_fl],
             yerr=sig_emp[mask_fl], fmt='^', color='red', ms=8,
             capsize=4, zorder=6, label='datos con sigma_floor (sigma_emp)', alpha=0.7)
ax1.errorbar(lt_all[mask_fl], r_all[mask_fl],
             yerr=sig_fl[mask_fl], fmt='^', color='red', ms=8,
             capsize=4, zorder=6, alpha=0.4)

# Líneas de referencia
ax1.axhline(GUE_REF, color='red', ls='-', lw=1.5, alpha=0.8, label=f'R_GUE = {GUE_REF}')
ax1.axhline(escenarios[0]['R'], color='navy', ls=':', lw=1, alpha=0.5)
ax1.axhline(escenarios[5]['R'], color='green', ls=':', lw=1, alpha=0.5)

ax1.set_ylabel('⟨r⟩', fontsize=13)
ax1.set_title('Tensión R∞: v6 sigma_floor vs versión anterior\n'
              'Triángulos rojos = puntos con sigma_floor aplicado (barras: emp vs fl)', fontsize=12)
ax1.legend(fontsize=7.5, loc='upper right', ncol=2)
ax1.set_ylim(0.5990, 0.6130)
ax1.grid(alpha=0.3)

# Panel residuos (escenario 1: v6 sigma_floor)
f0 = escenarios[0]
res_norm = f0['res'] / sig_fl
ax2.bar(lt_all[~fl_all], res_norm[~fl_all], color='royalblue', alpha=0.7, width=0.15, label='sigma_fl')
ax2.bar(lt_all[fl_all],  res_norm[fl_all],  color='red',       alpha=0.8, width=0.15, label='sigma_floor pts')
ax2.axhline(0, color='black', lw=1)
ax2.axhline( 2, color='gray', ls='--', lw=1, alpha=0.5)
ax2.axhline(-2, color='gray', ls='--', lw=1, alpha=0.5)
ax2.set_xlabel('log T', fontsize=12)
ax2.set_ylabel('residuo (σ)', fontsize=11)
ax2.set_title('Residuos modelo A (v6 sigma_floor)', fontsize=10)
ax2.legend(fontsize=9)
ax2.set_ylim(-3.5, 3.5)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../paper/figures/tension_Rinf_investigacion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en paper/figures/tension_Rinf_investigacion.png')

## 7. Conclusiones

In [ ]:
f_fl  = escenarios[0]  # v6 sigma_floor
f_emp = escenarios[1]  # v6 sigma_emp
f_nfl = escenarios[2]  # v6 sin puntos floor
f_v1  = escenarios[5]  # v1 viaje_completo

print('═'*70)
print('CONCLUSIONES: ¿Qué causa la tensión R∞?')
print('═'*70)

print(f"""
  v1 (12 pts, sigma=0.00060 uniforme):  R∞ = {f_v1['R']:.5f} ± {f_v1['eR']:.5f}  ({f_v1['nsig_gue']:+.1f}σ GUE)
  v6 sigma_floor (21 pts, ACTUAL):      R∞ = {f_fl['R']:.5f} ± {f_fl['eR']:.5f}  ({f_fl['nsig_gue']:+.1f}σ GUE)
  v6 sigma_emp   (sin floor):           R∞ = {f_emp['R']:.5f} ± {f_emp['eR']:.5f}  ({f_emp['nsig_gue']:+.1f}σ GUE)
  v6 sin 4 pts floor:                   R∞ = {f_nfl['R']:.5f} ± {f_nfl['eR']:.5f}  ({f_nfl['nsig_gue']:+.1f}σ GUE)
""")

print('  DESCOMPOSICIÓN DE LA TENSIÓN:')
delta_datos  = f_nfl['R'] - f_v1['R']  # efecto de nuevos datos (sin floor)
delta_floor  = f_fl['R'] - f_nfl['R']  # efecto del sigma_floor
delta_total  = f_fl['R'] - f_v1['R']
print(f'  ΔR∞ total (v6fl - v1)       = {delta_total:+.5f}')
print(f'  ΔR∞ nuevos datos (v6nfl-v1) = {delta_datos:+.5f}  ({delta_datos/delta_total*100:.0f}% del total)')
print(f'  ΔR∞ sigma_floor (v6fl-v6nfl)= {delta_floor:+.5f}  ({delta_floor/delta_total*100:.0f}% del total)')

print()
print('  DIAGNÓSTICO:')
print(f'  1. Los nuevos puntos en logT 19-24 genuinamente bajan R∞')
print(f'     (los datos de alta logT tienen ⟨r⟩ más cerca de GUE, pero aún por debajo)')
print(f'  2. El sigma_floor exacerba el efecto al dar peso extra a puntos bajos')
print(f'  3. El punto logT=20.2003 (r=0.60145) es el más influyente individualmente')
print()
print('  RECOMENDACIÓN para el paper:')
print(f'  Reportar AMBOS resultados:')
print(f'  - Con sigma_floor: R∞ = {f_fl["R"]:.5f} ± {f_fl["eR"]:.5f}  (conservador)')
print(f'  - Con sigma_emp:   R∞ = {f_emp["R"]:.5f} ± {f_emp["eR"]:.5f}  (optimista)')
print(f'  - La tensión con GUE es robusta en ambos casos ({f_fl["nsig_gue"]:+.1f}σ y {f_emp["nsig_gue"]:+.1f}σ)')
print()
print(f'  CONCLUSIÓN FÍSICA:')
print(f'  R∞ < GUE a logT≤24 es un resultado robusto, no un artefacto.')
print(f'  La convergencia al límite GUE es más lenta de lo esperado.')
print(f'  Para logT=24, el sistema todavía no ha alcanzado la asíntota GUE.')